# 스카우팅 분석 ① — 선수 위치 데이터 분석

`coordinates.csv`(영상에서 추출한 프레임별 피치 좌표)를 읽어, 선수별 **평균 위치 · 활동량 · 존 점유율**을 계산하고 시각화한다.

이 노트북에서 계산하는 지표는 스카우팅 브리프 §4의 CV 지표에 대응한다:
- **평균 위치** → §4-1(수신 위치), §4-4/5(전진성)
- **활동량/커버 범위** → §4-3(피지컬 엔진)
- **존 점유율** → §4-1/5(전진 vs 후방)

> 데이터는 분데스리가 샘플 클립 기준. 파이프라인 검증용이며, 이후 후보 선수 영상에 동일하게 적용한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
%matplotlib inline

# 피치 규격 (cm) — SoccerPitchConfiguration 기준
PITCH_L, PITCH_W = 12000, 7000

## Step 1. 데이터 불러오기 & 살펴보기
먼저 CSV를 읽고 구조를 확인한다. 어떤 역할(role)·팀(team)이 들어있는지 본다.

In [ ]:
df = pd.read_csv('data/coordinates.csv')
print('shape:', df.shape)
display(df.head())
print('\nrole 분포:'); print(df['role'].value_counts())
print('\nteam 분포:'); print(df['team'].value_counts(dropna=False))

## Step 2. 데이터 정제
선수(player)만 남기고, 호모그래피 변환 오차로 피치 밖(음수나 규격 초과)으로 튄 좌표는 제거한다.
실무 데이터엔 늘 이런 노이즈가 있어서, 분석 전 정제가 기본이다.

In [ ]:
players = df[df['role'] == 'player'].copy()
before = len(players)
players = players[
    (players['x'].between(0, PITCH_L)) &
    (players['y'].between(0, PITCH_W))
]
print(f'선수 행: {before} → {len(players)} (피치 밖 {before-len(players)}행 제거)')

## Step 3. 분석할 선수 고르기
`tracker_id`별 등장 프레임 수를 센다. **프레임이 많을수록 추적이 안정적**이라 분석이 믿을 만하다.

⚠️ 한계: `tracker_id`는 '추적 단위'라, 한 실제 선수가 가림(occlusion)·재등장 때문에 도중에 다른 ID로 끊길 수 있다.
완벽한 선수 단위 분석엔 재식별(re-ID)이 필요하지만, 지금은 가장 안정적인 ID 몇 개로 파이프라인을 검증한다.

In [ ]:
counts = players['tracker_id'].value_counts()
print('등장 프레임 수 상위:')
display(counts.head(10))

# 가장 안정적으로 추적된 선수 4명을 분석 대상으로
FOCUS_IDS = counts.head(4).index.tolist()
print('분석 대상 tracker_id:', FOCUS_IDS)

## Step 4. 지표 1 — 평균 위치 (§4-1, §4-4/5)
각 선수가 평균적으로 피치 어디에서 뛰었는지. x는 경기장 길이(0=자기 골문 쪽, 12000=상대 골문 쪽), y는 폭.

In [ ]:
mean_pos = players.groupby('tracker_id')[['x', 'y']].mean()
mean_pos = mean_pos.loc[FOCUS_IDS]
display(mean_pos.round(0))

## Step 5. 지표 2 — 활동량 / 커버 범위 (§4-3 피지컬 엔진)
- `std_x`, `std_y`: 위치 분산(클수록 넓게 움직임)
- `dist_m`: 총 이동 거리(m). 연속 프레임 간 거리를 합산. 카세미루가 잃은 '다리'를 보는 핵심 지표.

> 이동 거리는 호모그래피 노이즈로 과대 추정될 수 있어 절대값보다 **선수 간 상대 비교**로 본다.

In [ ]:
# 버전 안전한 방식: 정렬 후 그룹별 차분
ps = players.sort_values(['tracker_id', 'frame'])
ps['dx'] = ps.groupby('tracker_id')['x'].diff()
ps['dy'] = ps.groupby('tracker_id')['y'].diff()
ps['step'] = np.sqrt(ps['dx']**2 + ps['dy']**2)

cover = players.groupby('tracker_id').agg(std_x=('x', 'std'), std_y=('y', 'std'))
cover['dist_m'] = ps.groupby('tracker_id')['step'].sum() / 100.0  # cm -> m
display(cover.loc[FOCUS_IDS].round(1))

## Step 6. 지표 3 — 존 점유율 (§4-1, §4-5)
피치를 길이 방향으로 3등분(수비/중원/공격)해, 각 선수가 어느 존에 몇 % 머물렀는지.
앵커형이면 def/mid 비중이 높고, 전진형이면 att 비중이 높게 나온다.

In [ ]:
def zone(x):
    if x < PITCH_L / 3:        return 'def'
    elif x < PITCH_L * 2 / 3:  return 'mid'
    else:                      return 'att'

players['zone'] = players['x'].apply(zone)
zocc = (players.groupby('tracker_id')['zone']
        .value_counts(normalize=True)
        .unstack(fill_value=0) * 100)
display(zocc.loc[FOCUS_IDS][['def', 'mid', 'att']].round(0))

## Step 7. 피치 그리기 함수
시각화를 위해 matplotlib로 간단한 경기장을 그리는 헬퍼를 정의한다.

In [ ]:
def draw_pitch(ax):
    ax.add_patch(Rectangle((0, 0), PITCH_L, PITCH_W, fill=False, lw=2, color='white'))
    ax.plot([PITCH_L/2, PITCH_L/2], [0, PITCH_W], color='white', lw=1)
    ax.add_patch(Circle((PITCH_L/2, PITCH_W/2), 900, fill=False, lw=1, color='white'))
    ax.set_xlim(-300, PITCH_L+300); ax.set_ylim(-300, PITCH_W+300)
    ax.set_aspect('equal'); ax.set_facecolor('#3a7d3a')
    ax.set_xticks([]); ax.set_yticks([])
    return ax

## Step 8. 산점도 — 한 선수의 위치 분포
FOCUS_IDS 중 첫 선수의 모든 위치를 점으로, 평균 위치를 X로 표시.

In [ ]:
tid = FOCUS_IDS[0]
g = players[players['tracker_id'] == tid]
team = int(g['team'].iloc[0])

fig, ax = plt.subplots(figsize=(9, 5.5))
draw_pitch(ax)
ax.scatter(g['x'], g['y'], s=10, alpha=0.25, color='yellow')
ax.scatter([g['x'].mean()], [g['y'].mean()], s=250, color='red', marker='X',
           edgecolors='white', linewidths=1.5, zorder=5)
ax.set_title(f'Position scatter — tracker_id {tid} (team {team})', fontsize=12)
plt.show()

## Step 9. 히트맵 — 한 선수의 점유 밀도
같은 선수의 위치를 2D 히스토그램으로. 색이 밝을수록 그 구역에 오래 머물렀다는 뜻.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
draw_pitch(ax)
ax.hist2d(g['x'], g['y'], bins=[24, 14],
          range=[[0, PITCH_L], [0, PITCH_W]], cmap='hot', alpha=0.75)
ax.set_title(f'Heatmap — tracker_id {tid} (team {team})', fontsize=12)
plt.show()

## Step 10. 여러 선수 비교
FOCUS_IDS 선수들을 나란히 놓고 위치 성향을 비교한다. 누가 깊고, 누가 넓고, 누가 전진했는지 한눈에 보인다.

In [ ]:
fig, axes = plt.subplots(1, len(FOCUS_IDS), figsize=(4.2*len(FOCUS_IDS), 3.2))
for ax, t in zip(axes, FOCUS_IDS):
    draw_pitch(ax)
    gg = players[players['tracker_id'] == t]
    ax.scatter(gg['x'], gg['y'], s=5, alpha=0.2, color='yellow')
    ax.scatter([gg['x'].mean()], [gg['y'].mean()], s=120, color='red',
               marker='X', edgecolors='white', linewidths=1, zorder=5)
    ax.set_title(f'ID {t} (team {int(gg["team"].iloc[0])})', fontsize=10)
plt.tight_layout(); plt.show()

## Step 11. 해석 & 다음 단계

**지금 본 것** — 위치 데이터만으로 선수의 공간적 성향(어디서, 얼마나 넓게, 어느 존에서)을 정량화했다.
이게 브리프 §4-1·4-3·4-5의 CV 지표다.

**솔직한 한계 (포트폴리오에 명시할 것)**
1. 분데스리가 샘플 데이터 → 파이프라인 검증용. 실제 평가는 후보(페르난데스·토날리) 영상 필요.
2. `tracker_id` 파편화 → 정확한 선수 단위 분석엔 재식별(re-ID)이 필요. (SoccerNet의 player re-id 과제 영역)
3. 이동 거리는 호모그래피 노이즈로 과대 추정 가능 → 스무딩으로 보완 여지.

**다음 단계**
- 브리프 §4-2/4(압박저항·운반)는 위치만으론 부족 → FBref 이벤트 데이터 결합.
- 지표들을 0~1 정규화 → §5 가중치(수비30·압박저항25·체력20·운반15·전진10)로 적합도 점수 산출.
- 동일 파이프라인을 후보 영상에 적용 → 후보 비교 리포트.